# Fine-tune a 7B model on a free Colab GPU

This trains a real QLoRA adapter on a 7B-class model using Colab's free T4 GPU (16GB VRAM) - too big to fine-tune on a 4GB laptop GPU, but comfortable here.

**Before running:** In the Colab menu, go to `Runtime > Change runtime type` and select a `T4 GPU`, otherwise this will run on CPU and be extremely slow.

**Workflow:** upload a JSONL dataset built by `finetune/build_dataset.py` (or hand-written in the same `{"instruction", "input", "output"}` format) → train → download the resulting adapter → run `finetune/deploy_adapter.py` locally to load it into Ollama.

In [ ]:
!pip install -q unsloth

In [ ]:
from unsloth import FastLanguageModel

BASE_MODEL = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

## Upload your dataset

Build it locally first with:
```
python finetune/build_dataset.py "path/to/your/notes" --out finetune/data/dataset.jsonl
```
then upload `dataset.jsonl` below.

In [ ]:
from google.colab import files

uploaded = files.upload()
dataset_path = next(iter(uploaded.keys()))
print(f"using {dataset_path}")

In [ ]:
from datasets import load_dataset

PROMPT_TEMPLATE = """Below is an instruction that describes a task, paired with an input \
that provides further context. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}"""

dataset = load_dataset("json", data_files=dataset_path, split="train")
print(f"loaded {len(dataset)} examples")

def format_example(example):
    example["text"] = PROMPT_TEMPLATE.format(
        instruction=example["instruction"],
        input=example.get("input", ""),
        output=example["output"],
    ) + tokenizer.eos_token
    return example

dataset = dataset.map(format_example)

In [ ]:
from trl import SFTConfig, SFTTrainer

sft_config = SFTConfig(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    num_train_epochs=2,
    learning_rate=2e-4,
    logging_steps=1,
    optim="adamw_8bit",
    output_dir="checkpoints",
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=sft_config,
)

trainer.train()

## Save and download the adapter

This zips up the LoRA adapter and downloads it to your machine. Unzip it into `finetune/output/lora_adapter_7b/` in the project, then run:
```
python finetune/deploy_adapter.py finetune/output/lora_adapter_7b \
    --base-model-id Qwen/Qwen2.5-7B-Instruct \
    --ollama-base qwen2.5:7b \
    --name my-finetuned-7b
```
(pull the matching base into Ollama first: `ollama pull qwen2.5:7b`)

In [ ]:
import shutil

ADAPTER_DIR = "lora_adapter_7b"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

shutil.make_archive(ADAPTER_DIR, "zip", ADAPTER_DIR)
files.download(f"{ADAPTER_DIR}.zip")